# 3. Anomaly Detection and Port Clustering

Reads `processed_city_pairs.csv` from notebook 1, which -- unlike the original prototype -- already
contains the normalized columns this stage needs (they were computed once, centrally, in
`international_airlines.features.build_features`).

In [ ]:
import sys

sys.path.insert(0, '../src')

from pathlib import Path

import pandas as pd

from international_airlines.anomaly import cluster_ports, detect_anomalies

RESULTS_DIR = Path('../results')
df = pd.read_csv(RESULTS_DIR / 'processed_city_pairs.csv', parse_dates=['Date'])

df = detect_anomalies(df, contamination=0.05)
anomalies = df[df['Anomaly'] == -1]
print(f"Detected {len(anomalies)} anomalies out of {len(df)} rows")
anomalies[['Date', 'ForeignPort', 'Country', 'Passengers_Total', 'Freight_Total_(tonnes)', 'Passenger_Diff']].head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.scatterplot(x='Date', y='Passengers_Total', data=df, hue='Anomaly', palette={1: 'blue', -1: 'red'})
plt.title('Anomaly Detection in Passenger Traffic')
plt.show()

In [ ]:
port_clusters = cluster_ports(df, n_clusters=3)

plt.figure(figsize=(10, 6))
sns.scatterplot(x='Normalized_Passengers', y='Normalized_Freight', data=port_clusters, hue='Cluster', palette='viridis')
plt.title('Clustering of Foreign Ports by Traffic Behavior')
plt.show()

df.to_csv(RESULTS_DIR / 'final_processed_city_pairs.csv', index=False)
port_clusters.to_csv(RESULTS_DIR / 'port_clusters.csv', index=False)
port_clusters.sort_values('Cluster')